# Window/Hop Ablation - Qwen2-Audio-7B (4-bit)

In [ ]:
import sys, os, sqlite3, uuid, json, subprocess
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
for p in [ROOT] + list(ROOT.parents):
    if (p / '.gitignore').exists():
        ROOT = p; break
sys.path.insert(0, str(ROOT / 'backend/src'))
os.environ['PROJECT_ROOT'] = str(ROOT)

ABLATION_DIR = ROOT / 'data' / 'ablation_qwen2_audio'
VIDEO_DIR = ROOT / 'data/videos/eval'
GT_PATH = ROOT / 'data/videos/eval/ground_truth.xlsx'

import service.impl.audio_service_impl as asi
from service.impl.audio_service_impl import _merge_predictions, normalize
from utils.database import setup_database
import soundfile as sf

print(f'Project root: {ROOT}')


In [ ]:
# ── Audio taxonomy from the app DB (no hardcoded classes) ──
import sqlite3 as _sqlite3
import service.impl.audio_service_impl as _asi
from service.impl.config_store_service_impl import ConfigStoreServiceImpl as _CfgStore
_app_conn = _sqlite3.connect(str(ROOT / 'data/analysis.db'))
_app_conn.row_factory = _sqlite3.Row
_audio_tax = _CfgStore().get_section(_app_conn, 'audio_taxonomy') or {}
_app_conn.close()
_synonyms, _priority_pairs = _asi._build_synonym_lookup(
    _audio_tax.get('keywords') or {}, _audio_tax.get('priority') or [])
def normalize(text: str):
    return _asi.normalize(text, _synonyms, _priority_pairs)
print(f'Audio taxonomy loaded: {len(_audio_tax.get("classes") or [])} classes')


In [ ]:
# Grid
WINDOWS = [1.0, 2.5, 5.0, 10.0]
HOPS = [1, 2]
GRID = [(w, w/h) for w in WINDOWS for h in HOPS]
print(f'Grid: {len(GRID)} combos')


In [ ]:
expected_df = pd.read_excel(GT_PATH, sheet_name='Expected Events')
expected_df = expected_df.dropna(subset=['scene'])
expected_df['scene'] = expected_df['scene'].astype(int)
expected_df = expected_df[expected_df['audio'].notna() & (expected_df['audio'] != '')]
print(f'Expected events: {len(expected_df)}')


In [ ]:
import os, tempfile as tf, random
from transformers import Qwen2AudioForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
import torch

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(42)
random.seed(42)

model_id = 'Qwen/Qwen2-Audio-7B-Instruct'
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
processor = AutoProcessor.from_pretrained(model_id)
model = Qwen2AudioForConditionalGeneration.from_pretrained(
    model_id, device_map='auto', max_memory={0: '8GiB', 'cpu': '32GiB'},
    quantization_config=quant
)
print('Qwen2 loaded')

CLASSES = ['shout', 'impact', 'gunshot_or_explosion', 'engine',
          'tire_squeal', 'glass_breaking', 'horn', 'skidding']

def predict(clip, sr):
    p = os.path.join(tf.gettempdir(), 'q2ab.wav')
    sf.write(p, clip, sr, subtype='PCM_16')
    prompt = (
        "Analyze the audio clip. What is the most prominent sound?\n"
        "Choose exactly one category from: " + ", ".join(CLASSES) + "\n"
        "\n"
        "If NONE of the above categories apply, output: none()\n"
        "\n"
        "Do NOT output any other text."
    )
    conv = [{"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": [{"type": "audio", "audio_url": p}, {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
    inputs = processor(text=text, audio=clip, return_tensors='pt', padding=True, sampling_rate=sr).to(model.device)
    with torch.no_grad(): ids = model.generate(**inputs, max_new_tokens=32, do_sample=False)
    resp = processor.batch_decode(ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
    try: os.unlink(p)
    except: pass
    r = resp.strip().rstrip('. ').replace(' ', '_')
    return [(r, 1.0)] if r and r not in ('none', 'none()') else []

print('Model ready')


In [ ]:
from service.impl.events_service_impl import queries_for_condition
# ── Load GT audio once ──
gt_audio = pd.read_excel(GT_PATH, sheet_name='Ground Truth')
gt_audio = gt_audio.dropna(subset=['scene'])
gt_audio['scene'] = gt_audio['scene'].astype(int)
gt_audio = gt_audio[gt_audio['modality'] == 'audio']

def detect_fps(video_path: str, default: int = 24) -> int:
    try:
        probe = subprocess.check_output(
            ["ffprobe", "-v", "error", "-select_streams", "v:0",
             "-show_entries", "stream=avg_frame_rate,r_frame_rate",
             "-of", "json", video_path], timeout=10, stderr=subprocess.DEVNULL)
        info = json.loads(probe)
        for key in ("avg_frame_rate", "r_frame_rate"):
            fps_str = info["streams"][0].get(key, "")
            if fps_str and "/" in fps_str:
                num, den = fps_str.split("/")
                fps = int(num) // int(den) if int(den) else 0
                if fps > 0:
                    return fps
    except Exception:
        pass
    return default

FPS = detect_fps(str(next(VIDEO_DIR / f'scene{s}.mp4' for s in sorted(expected_df['scene'].unique()) if (VIDEO_DIR / f'scene{s}.mp4').exists())))
def t2f(t): return int(round(t * FPS))
def match_any(ivs, sf, ef):
    ms = [iv for iv in ivs if iv['end_frame'] >= sf and iv['start_frame'] <= ef]
    return max(ms, key=lambda iv: min(iv['end_frame'], ef) - max(iv['start_frame'], sf)) if ms else None
def ol(b, sf, ef): return min(b['end_frame'], ef) - max(b['start_frame'], sf) if b else 0


ANALYSIS_DIR = ROOT / 'data' / 'analysis_qwen2_audio'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
ABLATION_DIR = ROOT / 'data' / 'ablation_qwen2_audio'


DELTAS = {
    "delta_audio_fight": 1.0,  # 120 frames @24fps
}
RHOS = {
    "rho_audio_fight": 0.0,
    "rho_audio_vehicle_collision": 0.0,
    "rho_audio_vehicle_escape": 0.0,
}
ZETAS = {
    "zeta_audio_fight": '<=',
    "zeta_audio_vehicle_collision": '<=',
    "zeta_audio_vehicle_escape": '<=',
}

def build_params(deltas: dict, rhos: dict, zetas: dict, fps: int) -> dict:
    def frames(d: dict) -> dict:
        return {
            k: (round(v * fps) if isinstance(v, (int, float)) and not isinstance(v, bool) else v)
            for k, v in d.items()
        }
    return {**frames(deltas), **frames(rhos), **zetas}

def params_for_scene(scene) -> dict:
    return build_params(DELTAS, RHOS, ZETAS, detect_fps(str(VIDEO_DIR / f'scene{scene}.mp4')))
results = []
for combo_idx, (window, hop) in enumerate(GRID):
    print(f'[{combo_idx+1}/{len(GRID)}] Window={window}s, Hop={hop:.3f}s')
    win_label = f'w{window}s_h{hop}s'.replace('.', '_')
    db_path = ABLATION_DIR / f'qwen2_audio_{win_label}.db'
    if db_path.exists(): db_path.unlink()
    conn, cur = setup_database(db_path)
    asi.WINDOW_SECONDS = window
    asi.HOP_SECONDS = hop
    win_s = int(window * 16000)
    hop_s = int(hop * 16000)
    class_rows = []
    class_tp, class_fn, class_fp = {}, {}, {}
    event_rows = []
    Path('/tmp/eval_full_audio').mkdir(parents=True, exist_ok=True)
    for scene in sorted(expected_df['scene'].unique()):
        audio_path = Path('/tmp/eval_full_audio') / f'scene{scene}.wav'
        if not audio_path.exists():
            src = VIDEO_DIR / f'scene{scene}.mp4'
            if not src.exists(): continue
            subprocess.run(['ffmpeg', '-y', '-i', str(src), '-vn', '-acodec', 'pcm_s16le', '-ar', '16000', '-ac', '1', str(audio_path)], capture_output=True)
        waveform, sr = sf.read(str(audio_path), dtype='float32')
        if waveform.ndim > 1: waveform = waveform.mean(axis=-1)
        aid = f'ab{combo_idx}_s{scene}'
        existing = conn.execute(
            'SELECT COUNT(*) FROM AudioPerInterval WHERE AnalysisID = ?', (aid,)
        ).fetchone()[0]
        if existing > 0:
            print(f'    Scene {scene}: skipped ({existing} API)')
        else:
            all_res = []
            for start in range(0, max(1, len(waveform) - win_s + 1), hop_s):
                end = min(start + win_s, len(waveform))
                if end - start < sr // 2: continue
                sf_ = int(start / sr * FPS)
                ef_ = int(end / sr * 24)
                for lbl, conf in predict(waveform[start:end], sr):
                    lbl = normalize(lbl) or lbl
                    all_res.append({'audio_class': lbl, 'start_frame': sf_, 'end_frame': ef_, 'confidence': conf})
            intervals = _merge_predictions(all_res, fps=FPS, hop_seconds=hop)
            for iv in intervals:
                cur.execute('INSERT INTO AudioPerInterval (AnalysisID, AudioClass, StartFrame, EndFrame, Confidence) VALUES (?,?,?,?,?)',
                            (aid, iv['audio_class'], iv['start_frame'], iv['end_frame'], iv['confidence']))
            conn.commit()
        # Rebuild intervals from DB for skipped scenes
        rows = conn.execute(
            'SELECT AudioClass, StartFrame, EndFrame, Confidence FROM AudioPerInterval WHERE AnalysisID = ? ORDER BY StartFrame',
            (aid,)
        ).fetchall()
        intervals = [{'audio_class': r[0], 'start_frame': r[1], 'end_frame': r[2], 'confidence': r[3]} for r in rows]
        # Class-level capture (blind: best overlapping any-class prediction)
        for _, gt in gt_audio[gt_audio['scene'] == scene].iterrows():
            gt_sf, gt_ef = t2f(gt['starttime']), t2f(gt['endtime'])
            top = match_any(intervals, gt_sf, gt_ef)
            class_rows.append({'scene': scene, 'gt_class': gt['class'], 'gt_start': gt_sf, 'gt_end': gt_ef,
                              'pred_class': top['audio_class'] if top else '', 'pred_start': top['start_frame'] if top else '',
                              'pred_end': top['end_frame'] if top else '', 'overlap_frames': ol(top, gt_sf, gt_ef), 'correct': bool(top and top['audio_class'] == gt['class'])})
        # Event-level capture
        sql_map = queries_for_condition("B", params_for_scene(scene), analysis_id=aid)
        for _, row in expected_df[expected_df['scene'] == scene].iterrows():
            evt = row['event']
            sql = sql_map.get(evt, 'SELECT 0 WHERE 1=0')
            det = not pd.read_sql_query(sql, conn).empty
            event_rows.append({'scene': scene, 'event': evt,
                               'detected': 'YES' if det else 'NO', 'result': 'TP' if det else 'FN',
                               'audio': ', '.join(f"{iv['audio_class']}({iv['start_frame']}-{iv['end_frame']})" for iv in intervals)})
        # Accumulate detection metrics (blind: any-overlap = TP, no-overlap = FN; same-class pred without GT overlap = FP)
        for _, gt in gt_audio[gt_audio['scene'] == scene].iterrows():
            cls = gt['class']
            gts, gte = t2f(gt['starttime']), t2f(gt['endtime'])
            if any(iv['audio_class'] == cls and iv['end_frame'] >= gts and iv['start_frame'] <= gte for iv in intervals):
                class_tp[cls] = class_tp.get(cls, 0) + 1
            else:
                class_fn[cls] = class_fn.get(cls, 0) + 1
        for iv in intervals:
            cls = iv['audio_class']
            overlaps_any_gt = any(
                t2f(gt['starttime']) <= iv['end_frame'] and t2f(gt['endtime']) >= iv['start_frame'] and gt['class'] == cls
                for _, gt in gt_audio[gt_audio['scene'] == scene].iterrows()
            )
            if not overlaps_any_gt:
                class_fp[cls] = class_fp.get(cls, 0) + 1
    # False Positive pass: check scenes where event should NOT happen
    audio_scenes = set(gt_audio['scene'].unique())
    for evt in sorted(expected_df['event'].unique()):
        pos_scenes = set(expected_df[expected_df['event'] == evt]['scene'])
        aid = f'ab{combo_idx}_s{{scene}}'
        for neg_scene in sorted(expected_df['scene'].unique()):
            if neg_scene in pos_scenes:
                continue
            if neg_scene not in audio_scenes:
                continue
            aid = f'ab{combo_idx}_s{neg_scene}'
            sql_map = queries_for_condition("B", params_for_scene(neg_scene), analysis_id=aid)
            sql = sql_map.get(evt, 'SELECT 0 WHERE 1=0')
            try:
                df = pd.read_sql_query(sql, conn)
                det = not df.empty
                if det:
                    rows = conn.execute(
                        'SELECT AudioClass, StartFrame, EndFrame FROM AudioPerInterval WHERE AnalysisID = ?', (aid,)
                    ).fetchall()
                    audio = ', '.join(f'{r[0]}({r[1]}-{r[2]})' for r in rows)
                    event_rows.append({'scene': neg_scene, 'event': evt,
                        'detected': 'YES', 'result': 'FP', 'audio': audio})
            except:
                pass

    api = conn.execute('SELECT COUNT(*) FROM AudioPerInterval').fetchone()[0]
    tp = sum(1 for r in event_rows if r['result'] == 'TP')
    fp = sum(1 for r in event_rows if r['result'] == 'FP')
    fn = sum(1 for r in event_rows if r['result'] == 'FN')
    support = tp + fn
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    print(f'  API={api}, P={precision:.3f} R={recall:.3f} F1={f1:.3f} TP={tp} FP={fp} FN={fn}')
    results.append({'audio': 'qwen2_audio', 'window': window, 'hop': round(hop, 3), 'precision': round(precision, 3), 'recall': round(recall, 3), 'f1': round(f1, 3), 'TP': tp, 'FP': fp, 'FN': fn, 'support': support, 'API': api})
    conn.close()
    # Write xlsx for this combo
    method = f'qwen2_audio_{win_label}'
    pdf = pd.DataFrame(class_rows)
    with pd.ExcelWriter(ANALYSIS_DIR / f'audio_class_eval_{method}.xlsx') as writer:
        metrics = []
        for cls in sorted(class_tp.keys() | class_fn.keys() | class_fp.keys()):
            tp = class_tp.get(cls, 0)
            fn = class_fn.get(cls, 0)
            fp = class_fp.get(cls, 0)
            support = tp + fn
            p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
            metrics.append({'class': cls, 'precision': round(p, 3), 'recall': round(r, 3), 'f1': round(f1, 3), 'TP': tp, 'FP': fp, 'FN': fn, 'support': support})
        pd.DataFrame(metrics).to_excel(writer, sheet_name='Summary', index=False)
        for sc in sorted(expected_df['scene'].unique()):
            sc_df = pdf[pdf['scene'] == sc]
            if not sc_df.empty: sc_df.to_excel(writer, sheet_name=f'Scene_{sc}', index=False)
    edf = pd.DataFrame(event_rows)
    with pd.ExcelWriter(ANALYSIS_DIR / f'audio_event_eval_{method}.xlsx') as writer:
        metrics = []
        for evt in sorted(edf['event'].unique()):
            sub = edf[edf['event'] == evt]
            tpp = len(sub[sub['result'] == 'TP'])
            fpp = len(sub[sub['result'] == 'FP'])
            fnn = len(sub[sub['result'] == 'FN'])
            support = tpp + fnn
            p = tpp / (tpp + fpp) if (tpp + fpp) > 0 else 0.0
            r = tpp / (tpp + fnn) if (tpp + fnn) > 0 else 0.0
            f1 = 2*p*r/(p+r) if (p+r) > 0 else 0.0
            metrics.append({'event': evt, 'precision': round(p,3), 'recall': round(r,3), 'f1': round(f1,3), 'TP': tpp, 'FP': fpp, 'FN': fnn, 'support': support})
        pd.DataFrame(metrics).to_excel(writer, sheet_name='Summary', index=False)
        for sc in sorted(expected_df['scene'].unique()):
            sc_df = edf[edf['scene'] == sc]
            if not sc_df.empty: sc_df.to_excel(writer, sheet_name=f'Scene_{sc}', index=False)
results_df = pd.DataFrame(results)
results_df.to_excel(ROOT / 'data' / 'analysis_qwen2_audio' / 'summary.xlsx', index=False)
print(f'Results saved to ablation_results.xlsx')
display(results_df)
